# Phase 3-6: Predicting Online Community Collapse
This notebook calculates the 'Collapse' target variable mathematically, engineers time-lagged features (to predict collapse before it happens), and evaluates multiple machine learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load Preprocessed Data
df = pd.read_csv('../data/processed_subreddit_weekly_enriched.csv')
df['year_week'] = pd.to_datetime(df['year_week'])
df = df.sort_values(by=['subreddit', 'year_week']).reset_index(drop=True)
print(f"Data Loaded: {df.shape[0]} rows, {df.shape[1]} columns.")

### Step 1. Define the "Collapse" Metric (Target Variable `y`)
We define engagement = `total_comments` + `post_count`. 
A community is "At Risk/Collapsed" (1) if its 4-week rolling engagement drops below **30%** of its historical peak. Otherwise, it is Healthy (0).

In [ ]:
df['engagement'] = df['total_comments'] + df['post_count']

y_labels = []
for sub in df['subreddit'].unique():
    sub_data = df[df['subreddit'] == sub].copy()
    # Calculate historical peak up to current week (Expanding Max)
    historical_peak = sub_data['engagement'].expanding().max()
    
    # Calculate 4-week rolling average of engagement to smooth out outliers
    rolling_avg = sub_data['engagement'].rolling(window=4, min_periods=1).mean()
    
    # Condition: Is rolling_avg < 30% of the historical peak?
    collapsed_mask = (rolling_avg < (0.30 * historical_peak)) & (historical_peak > 50) # Ignore tiny subreddits
    sub_data['is_collapsed'] = collapsed_mask.astype(int)
    y_labels.append(sub_data)

df = pd.concat(y_labels)
print("Target Variable Created!")
print(df['is_collapsed'].value_counts(normalize=True))

### Step 2. Feature Engineering (Time Windowing / Lags)
To predict failure **before** it happens, we cannot use data from Week $T$ to predict Week $T$. 
We must use data from weeks $T-1, T-2, T-3, T-4$ to predict the state of Week $T$.

In [ ]:
feature_cols = ['post_count', 'total_comments', 'avg_score', 'avg_upvote_ratio',
                'total_awards', 'total_crossposts', 'avg_subscribers']

# Create lagged features for 1, 2, 3, and 4 weeks ago
for col in feature_cols:
    for lag in [1, 2, 3, 4]:
        df[f'{col}_lag{lag}'] = df.groupby('subreddit')[col].shift(lag)

# Drop rows with NaN (the first 4 weeks of every subreddit won't have lag data)
df_model = df.dropna().copy()
print(f"Modeling dataset shape after windowing: {df_model.shape}")

### Step 3. Train-Test Split (Temporal)
We must strictly sort by time. We train on the past to predict the future. Random shuffling would cause "Data Leakage".

In [ ]:
df_model = df_model.sort_values(by='year_week')

# Select Features and Target
predictors = [col for col in df_model.columns if 'lag' in col]
X = df_model[predictors]
y = df_model['is_collapsed']

# 80% Train, 20% Test Split chronologically
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training instances: {X_train_scaled.shape[0]}, Test instances: {X_test_scaled.shape[0]}")

### Step 4. Train and Evaluate Models
We will train 3 models:
1. **Logistic Regression** (Baseline, tests linear boundaries)
2. **Random Forest** (Captures non-linear behavioral proxies)
3. **Gradient Boosting / XGBoost Proxy** (State of the art for imbalanced tabular data)

In [ ]:
models = {
    "Baseline: Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Advanced: Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "SOTA: Gradient Boosting": HistGradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n{'='*40}\nTraining {name}...\n{'='*40}")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    if name == "Advanced: Random Forest":
        rf_model = model # Save for feature importance analysis
    if name == "SOTA: Gradient Boosting":
        gb_model = model


### Step 5. Interpretability: What predicts a collapse?
Let's look at which features the Random Forest found most important in identifying a future collapse.

In [ ]:
importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': predictors, 'Importance': importances}).sort_values('Importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df.head(10))
plt.title('Top 10 Early Warning Signs (Features) for Community Collapse')
plt.show()

### VISUALIZATION 1: The Anatomy of a Collapse (The Real-World View)
Let's look at the lifecycle of a specific subreddit to visualize when the math flags a 'Collapse'.

In [ ]:
# Find a subreddit that has a collapse
df_clean = df.dropna().sort_values(by='year_week').copy()
collapsed_subs = df_clean[df_clean['is_collapsed'] == 1]['subreddit'].unique()
target_sub = collapsed_subs[0] if len(collapsed_subs) > 0 else df_clean['subreddit'].unique()[0]
sub_data = df_clean[df_clean['subreddit'] == target_sub].copy()

plt.figure(figsize=(14, 6))
plt.plot(sub_data['year_week'], sub_data['engagement'], label='Actual Engagement (Weekly)', color='blue', alpha=0.6)

historical_peak = sub_data['engagement'].expanding().max()
plt.plot(sub_data['year_week'], historical_peak, label='Historical Peak Normal', color='green', linestyle='--', alpha=0.7)

collapse_points = sub_data[sub_data['is_collapsed'] == 1]
plt.scatter(collapse_points['year_week'], collapse_points['engagement'], color='red', label='Flagged as Collapsed', zorder=5)

plt.title(f'Anatomy of a Collapse: r/{target_sub}')
plt.xlabel('Date')
plt.ylabel('Engagement (Comments + Posts)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### VISUALIZATION 2: Predictive Early Warning Radar
Plotting the model's *Probability of Imminent Collapse* over time compared to the actual collapse event.

In [ ]:
sub_features = sub_data[predictors]
sub_features_scaled = scaler.transform(sub_features)
sub_probs = gb_model.predict_proba(sub_features_scaled)[:, 1]

plt.figure(figsize=(14, 6))
plt.plot(sub_data['year_week'], sub_probs, label='Predicted Probability of Collapse', color='purple', linewidth=2)
plt.axhline(y=0.5, color='orange', linestyle='--', label='50% Warning Threshold')

for _, row in collapse_points.iterrows():
    plt.axvline(x=row['year_week'], color='red', alpha=0.1)
if len(collapse_points) > 0:
    plt.axvspan(collapse_points['year_week'].min(), collapse_points['year_week'].max(), color='red', alpha=0.2, label='Actual Collapse Period')

plt.title(f'Predictive Early Warning Radar: r/{target_sub}')
plt.xlabel('Date')
plt.ylabel('Probability of Collapse')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### VISUALIZATION 3: Behavioral Degradation (What Dies First?)
Comparing the quality of human interaction between Healthy and Collapsed states.

In [ ]:
behavioral_cols = ['total_awards', 'total_crossposts', 'avg_score']
grouped = df.groupby('is_collapsed')[behavioral_cols].mean().reset_index()

from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler()
grouped_scaled = grouped.copy()
grouped_scaled[behavioral_cols] = scaler_minmax.fit_transform(grouped[behavioral_cols].T).T # Normalize for visual comparison
melted = grouped_scaled.melt(id_vars='is_collapsed', value_vars=behavioral_cols)
melted['is_collapsed'] = melted['is_collapsed'].map({0: 'Healthy', 1: 'Collapsed'})

plt.figure(figsize=(10, 6))
sns.barplot(data=melted, x='variable', y='value', hue='is_collapsed', palette=['#2ecc71', '#e74c3c'])
plt.title('Behavioral Degradation: Healthy vs Collapsed States (Normalized)')
plt.ylabel('Relative Frequency / Quality')
plt.xlabel('Behavioral Proxy')
plt.legend(title='Community State')
plt.show()

### VISUALIZATION 4: Production ML Metrics (Confusion Matrix & ROC)
Evaluating the Gradient Boosting classifier's performance mathematically.

In [ ]:
from sklearn.metrics import roc_curve, auc, RocCurveDisplay

gb_pred = gb_model.predict(X_test_scaled)
gb_probs = gb_model.predict_proba(X_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, gb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Predicted Healthy', 'Predicted Collapsed'],
            yticklabels=['Actual Healthy', 'Actual Collapsed'])
axes[0].set_title('Confusion Matrix (Gradient Boosting)')

# ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, gb_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc="lower right")

plt.show()

### VISUALIZATION 5: SHAP Values (Explainable AI)
Understanding individual predictions using SHapley Additive exPlanations. This proves *why* the model made specific flags.

In [ ]:
import shap
shap.initjs()

# Sample test set for performance
sample_idx = np.random.choice(X_test_scaled.shape[0], min(200, X_test_scaled.shape[0]), replace=False)
X_test_sample = X_test_scaled[sample_idx]

explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test_sample)

# Global feature importance
plt.figure()
plt.title('SHAP Summary Plot (Global Interpretability)')
shap.summary_plot(shap_values, features=X_test_sample, feature_names=predictors, show=False)
plt.show()
